# Day 19 — Pandas transformation: groupby, agg, pivot_table, melt
Objectives:
- Summarize with groupby and custom aggregations.
- Reshape data with pivot_table and melt.
- Combine multiple metrics and format outputs.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-19`. Read
`python/ds-60day/companion-guides/day19_pandas_groupby_pivot.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Grouped analysis starts with grain. `groupby` splits rows by key values,
applies reductions or transformations, and combines results. An
aggregation reduces each group and changes row grain; `transform`
returns one aligned value per original row and preserves row count.

A pivot makes combinations of keys into rows and columns while applying
an aggregation for duplicates. `melt` turns wide measurement columns
into a long variable/value pair. Reshaping changes representation, so
state which columns identify an observation and reconcile totals before
and after.

### Vocabulary

- **group key:** the column values defining membership in a group.
- **aggregation:** a reduction from many rows to a summary.
- **transform:** a group calculation aligned back to every original row.
- **pivot:** a reshape that places one key's values across columns.
- **melt:** a reshape from wide measurement columns to long rows.
- **grain:** what one output row represents after an operation.

## Syntax anatomy

`.groupby("region", as_index=False).agg(total=("amount", "sum"),
orders=("order_id", "nunique"))` names the group key, keeps it as a
column, and uses named aggregations whose left sides are output column
names. In `pivot_table`, `index` defines output rows, `columns` defines
output columns, `values` supplies measurements, and `aggfunc` resolves
duplicate combinations.

### Worked example 1 — Aggregate to one row per group

Write the output grain before reading the result. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "region": ["east", "east", "west", "west"],
    "amount": [10, 15, 7, 8],
})
summary = (
    orders.groupby("region", as_index=False)
    .agg(order_count=("order_id", "nunique"), total=("amount", "sum"))
)
summary.to_dict("records")

**Expected observation:** `[{'region': 'east', 'order_count': 2, 'total': 25}, {'region': 'west', 'order_count': 2, 'total': 15}]`.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Align a group total back to every source row

Transform preserves the original index and row count. Predict first; then run the next cell.

In [ ]:
orders = orders.assign(
    region_total=orders.groupby("region")["amount"].transform("sum")
)
orders[["order_id", "region", "amount", "region_total"]].to_dict("records")

**Expected observation:** Every east row receives `25` and every west row receives `15`; there are still four rows.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. State input and output grain before deciding between `agg` and `transform`.
2. Use named aggregations instead of accepting confusing MultiIndex columns.
3. Pass an explicit `aggfunc` and `fill_value` policy to `pivot_table`.
4. Reconcile additive totals and row counts across a reshape.

**Alternative to compare:** Use `crosstab` for frequency tables, `pivot` only when combinations are unique, and `pivot_table` when duplicates need aggregation.

**Boundary to test:** Missing group keys, unobserved categories, duplicate pivot cells, all-missing groups, and non-additive measures need explicit handling.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import pandas as pd, numpy as np, seaborn as sns
df = sns.load_dataset('tips')
# Groupby with multiple aggregations
agg = df.groupby(['day','time']).agg(
    total_bill_mean=('total_bill','mean'),
    tip_mean=('tip','mean'),
    count=('total_bill','size'),
)
agg.reset_index().head()

# Pivot table
pt = pd.pivot_table(df, values='tip', index='day', columns='time', aggfunc='mean')
pt

# Melt back to long
long = pt.reset_index().melt(id_vars='day', var_name='time', value_name='avg_tip')
long.head()


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Group the lesson data by two categorical columns and compute named count, sum, mean, and median outputs. **Before code:** state the output grain and whether missing group keys are included. **Constraints:** use named `.agg`, avoid ambiguous MultiIndex columns, and distinguish row count from unique-entity count.
   **Verify:** assert group-key uniqueness and reconcile the summed additive measure to the input total.

2. Create a pivot table and then return it to long form with `melt` or `stack`. **Contract:** choose explicit index, column, value, aggregation, and missing-cell policy.
   **Expected behavior:** the long form clearly identifies every dimension and measurement.
   **Verify:** reconcile non-missing values/totals and explain any rows introduced or removed.

### Additional mastery practice

Write the input and output grain before grouping or reshaping. Reconcile row counts and totals at equivalent grains.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict whether rows with a missing group key are included by default and compare `dropna=True` with `dropna=False`.
   **Progressive hint:** Missing group keys are normally excluded unless requested.
   **Verify:** Group the same fixture both ways; assert totals differ exactly by the missing-key rows and state which output matches the intended policy.
4. **Tracing:** Trace a wide table through `melt` and back through `pivot_table`; name the identifier, variable, and value columns.
   **Progressive hint:** Reshaping changes layout, not the underlying measures.
   **Verify:** Assert the long table has identifier/variable/value columns and that pivoting back preserves the original labeled values and additive total.
5. **Implementation:** Implement grouped weighted means without using a simple mean of group means.
   **Progressive hint:** Aggregate weighted numerators and denominators at the same grain.
   **Verify:** For two groups, compute numerator and denominator explicitly and assert the grouped weighted means, including the chosen zero-weight behavior.
6. **Debugging:** Repair a calculation that joins a customer-level total back to line items and then sums it, multiplying totals by line count.
   **Progressive hint:** Do not re-aggregate a measure after broadcasting it to a finer grain.
   **Verify:** Show the inflated total after summing broadcast customer totals, then assert the repaired grain-aware calculation matches the original customer-level total.
7. **Edge case and explanation:** Handle groups whose total weight is zero and categories with no observed rows; state whether they appear in output.
   **Progressive hint:** Make zero-denominator and categorical `observed` behavior explicit.
   **Verify:** Test a zero-weight group and an unobserved categorical level; assert their value/presence follows the documented denominator and `observed` policy.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Group the lesson data by two categorical columns and compute named count, sum, mean, and median outputs. **Before code:** state the output grain and whether missing group keys are included. **Constraints:** use named `.agg`, avoid ambiguous MultiIndex columns, and distinguish row count from unique-entity count. **Verify:** assert group-key uniqueness and reconcile the summed additive measure to the input total.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Group the lesson data by two categorical columns and compute named count, sum, mean, and median outputs. state the output grain and whether missing group keys are included. use...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Create a pivot table and then return it to long form with `melt` or `stack`. **Contract:** choose explicit index, column, value, aggregation, and missing-cell policy. **Expected behavior:** the long form clearly identifies every dimension and measurement. **Verify:** reconcile non-missing values/totals and explain any rows introduced or removed.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Create a pivot table and then return it to long form with `melt` or `stack`. choose explicit index, column, value, aggregation, and missing-cell policy. the long form clearly id...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict whether rows with a missing group key are included by default and compare `dropna=True` with `dropna=False`. **Progressive hint:** Missing group keys are normally excluded unless requested. **Verify:** Group the same fixture both ways; assert totals differ exactly by the missing-key rows and state which output matches the intended policy.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict whether rows with a missing group key are included by default and compare `dropna=True` with `dropna=False`. Missing group keys are normally excluded unless requested. G...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace a wide table through `melt` and back through `pivot_table`; name the identifier, variable, and value columns. **Progressive hint:** Reshaping changes layout, not the underlying measures. **Verify:** Assert the long table has identifier/variable/value columns and that pivoting back preserves the original labeled values and additive total.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace a wide table through `melt` and back through `pivot_table`; name the identifier, variable, and value columns. Reshaping changes layout, not the underlying measures. Assert...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement grouped weighted means without using a simple mean of group means. **Progressive hint:** Aggregate weighted numerators and denominators at the same grain. **Verify:** For two groups, compute numerator and denominator explicitly and assert the grouped weighted means, including the chosen zero-weight behavior.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Implement grouped weighted means without using a simple mean of group means. Aggregate weighted numerators and denominators at the same grain. For two groups, compute numerator...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a calculation that joins a customer-level total back to line items and then sums it, multiplying totals by line count. **Progressive hint:** Do not re-aggregate a measure after broadcasting it to a finer grain. **Verify:** Show the inflated total after summing broadcast customer totals, then assert the repaired grain-aware calculation matches the original customer-level total.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair a calculation that joins a customer-level total back to line items and then sums it, multiplying totals by line count. Do not re-aggregate a measure after broadcasting it...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Handle groups whose total weight is zero and categories with no observed rows; state whether they appear in output. **Progressive hint:** Make zero-denominator and categorical `observed` behavior explicit. **Verify:** Test a zero-weight group and an unobserved categorical level; assert their value/presence follows the documented denominator and `observed` policy.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Handle groups whose total weight is zero and categories with no observed rows; state whether they appear in output. Make zero-denominator and categorical `observed` behavior exp...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
